# Oscillon MG — Post-run analysis

Read simulation data produced by `RunScripts/run_oscillon.py` and plot
oscillon diagnostics (compactness, density contrast, energy-density profiles).

Each run lives in a subfolder of this `DATA/` directory, named by its parameter tag.

In [ ]:
import sys, os, glob
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

from core.grid import Grid
from core.spacing import CubicSpacing
from core.statevector import StateVector
from matter.scalarmatter_MG import ScalarMatter
from backgrounds.sphericalbackground import FlatSphericalBackground
from bssn.oscillondiagnostic import get_oscillon_diagnostic

DATA_DIR = os.path.dirname(os.path.abspath("__file__"))
if not os.path.isabs(DATA_DIR):
    DATA_DIR = os.getcwd()
print(f"DATA_DIR = {DATA_DIR}")

## 1 — Discover available runs

In [ ]:
run_dirs = sorted([
    d for d in glob.glob(os.path.join(DATA_DIR, "lgb*"))
    if os.path.isdir(d) and os.path.exists(os.path.join(d, "solution.npy"))
])

print(f"Found {len(run_dirs)} completed run(s):")
for d in run_dirs:
    print(f"  {os.path.basename(d)}")

## 2 — Load data and compute diagnostics

In [ ]:
# Default physics parameters (must match what was used in run_oscillon.py)
scalar_mu = 1
selfinteraction = 0.08
chi0 = 0.15
coupling = "quadratic"
a_mg = 0.2
b_mg = 0.4

r_max = 150
min_dr = 1 / 16
max_dr = 2

# Reconstruct grid (same for all runs with these grid params)
matter_ref = ScalarMatter(scalar_mu, selfinteraction)
sv_ref = StateVector(matter_ref)
spacing_ref = CubicSpacing(**CubicSpacing.get_parameters(r_max, min_dr, max_dr))
grid_ref = Grid(spacing_ref, sv_ref)
bg_ref = FlatSphericalBackground(grid_ref.r)

print(f"Grid: {grid_ref.r.size} points,  r in [{grid_ref.r[0]:.4f}, {grid_ref.r[-1]:.1f}]")

In [ ]:
results = {}  # lgb_value -> dict with 'osc', 'sol', 't', 'tag', 'run_dir'

for run_dir in run_dirs:
    tag = os.path.basename(run_dir)

    # Extract lambda_GB from folder name
    lgb = float(tag.split("_")[0].replace("lgb", ""))

    sol = np.load(os.path.join(run_dir, "solution.npy"))
    t   = np.load(os.path.join(run_dir, "t.npy"))

    # Try pre-computed diagnostics first
    diag_path = os.path.join(run_dir, "diagnostics.npz")
    if os.path.exists(diag_path):
        osc = dict(np.load(diag_path, allow_pickle=True))
        print(f"lgb={lgb} — loaded pre-computed diagnostics")
    else:
        print(f"lgb={lgb} — computing diagnostics ...")
        params = (lgb, a_mg, b_mg, chi0, coupling)
        osc = get_oscillon_diagnostic(
            sol, t, grid_ref, bg_ref,
            ScalarMatter(scalar_mu, selfinteraction),
            params, surface_threshold=0.05, r_max_diag=100.0,
        )
        np.savez(os.path.join(run_dir, "diagnostics.npz"), **osc)

    results[lgb] = dict(osc=osc, t=t, tag=tag, run_dir=run_dir)
    C_max = np.max(osc["C"])
    print(f"  lgb={lgb:.3f}  max compactness C = {C_max:.6e}")

    del sol

## 3 — Compactness vs $\lambda_{\rm GB}$

In [ ]:
lambdas_sorted = np.array(sorted(results.keys()))
C_max_vals = np.array([np.max(results[l]["osc"]["C"]) for l in lambdas_sorted])

# Normalise to GR if available, otherwise to first run
C_ref = C_max_vals[0] if 0.0 in results else C_max_vals[0]
if 0.0 in results:
    C_ref = np.max(results[0.0]["osc"]["C"])
C_norm = C_max_vals / C_ref

fig, ax = plt.subplots(figsize=(8, 5.5))

if 0.0 in results:
    ax.axhline(1.0, color="#4E9BD9", lw=3.5, label="GR", zorder=2)
    mg_mask = lambdas_sorted > 0
else:
    mg_mask = np.ones_like(lambdas_sorted, dtype=bool)

ax.plot(lambdas_sorted[mg_mask], C_norm[mg_mask],
        "o-", color="#D94E2B", lw=3, ms=8, zorder=3, label="EsGB")

ax.set_xlabel(r"$\lambda_{\rm GB}$", fontsize=16)
ax.set_ylabel(r"$\mathcal{C}\;/\;\mathcal{C}_{\rm ref}$", fontsize=16)
ax.set_title("Oscillon Compactness", fontsize=16, fontweight="bold")
ax.legend(fontsize=14, loc="upper left")
ax.tick_params(labelsize=12)
ax.set_xlim(left=0)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

## 4 — Compactness time series

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(results)))

for i, lgb in enumerate(sorted(results.keys())):
    osc = results[lgb]["osc"]
    label = f"$\\lambda_{{\\rm GB}}={lgb}$"
    if lgb == 0.0:
        label = r"GR ($\lambda_{\rm GB}=0$)"
    ax.plot(osc["ln_a"], osc["C"] / C_ref,
            color=colors[i], lw=2, label=label)

ax.set_xlabel(r"$\ln(a)$", fontsize=14)
ax.set_ylabel(r"$\mathcal{C}\;/\;\mathcal{C}_{\rm ref}^{\rm max}$", fontsize=14)
ax.set_title("Compactness evolution", fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(1.5, 3.5)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

## 5 — Density contrast evolution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(results)))

for i, lgb in enumerate(sorted(results.keys())):
    osc = results[lgb]["osc"]
    dc = osc["delta_c"]
    pos = dc > 0
    y = np.full_like(dc, np.nan)
    y[pos] = np.log10(dc[pos])

    label = f"$\\lambda_{{\\rm GB}}={lgb}$"
    if lgb == 0.0:
        label = r"GR"
    ax.plot(osc["ln_a"], y, color=colors[i], lw=1.5, label=label)

ax.set_xlabel(r"$\ln(a)$", fontsize=14)
ax.set_ylabel(r"$\log_{10}(\delta_c)$", fontsize=14)
ax.set_title("Density contrast growth", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

## 6 — Central density and Hubble rate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(results)))

for i, lgb in enumerate(sorted(results.keys())):
    osc = results[lgb]["osc"]
    label = f"$\\lambda_{{\\rm GB}}={lgb}$"

    # Central density
    rc = osc["rho_c"]
    pos = rc > 0
    y = np.full_like(rc, np.nan)
    y[pos] = np.log10(rc[pos])
    axes[0].plot(osc["ln_a"], y, color=colors[i], lw=1.5, label=label)

    # Hubble rate
    H = -osc["K_avg"] / 3.0
    axes[1].plot(osc["ln_a"], H, color=colors[i], lw=1.5, label=label)

axes[0].set_xlabel(r"$\ln(a)$", fontsize=13)
axes[0].set_ylabel(r"$\log_{10}(\rho_c)$", fontsize=13)
axes[0].set_title("Central energy density", fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.25)

axes[1].set_xlabel(r"$\ln(a)$", fontsize=13)
axes[1].set_ylabel(r"$H = -\langle K \rangle / 3$", fontsize=13)
axes[1].set_title("Hubble parameter", fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.25)

fig.tight_layout()
plt.show()

## 7 — Full diagnostic panels (single run)

Select a specific run by its `lambda_GB` value to see the four-panel diagnostic figure.

In [ ]:
from bssn.oscillondiagnostic import plot_paper_diagnostics

# Pick a run to inspect in detail
lgb_inspect = sorted(results.keys())[-1]  # last (largest) coupling
print(f"Showing diagnostics for lambda_GB = {lgb_inspect}")

fig, axes = plot_paper_diagnostics(results[lgb_inspect]["osc"])
fig.suptitle(rf"$\lambda_{{\rm GB}} = {lgb_inspect}$", fontsize=15)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 8 — Energy-density radial profiles

In [ ]:
from bssn.oscillondiagnostic import plot_density_profiles_at_times

# Reload the full solution for the selected run to make spatial profiles
run_dir_inspect = results[lgb_inspect]["run_dir"]
sol_inspect = np.load(os.path.join(run_dir_inspect, "solution.npy"))
t_inspect = results[lgb_inspect]["t"]

params_inspect = (lgb_inspect, a_mg, b_mg, chi0, coupling)
osc_inspect = get_oscillon_diagnostic(
    sol_inspect, t_inspect, grid_ref, bg_ref,
    ScalarMatter(scalar_mu, selfinteraction),
    params_inspect, surface_threshold=0.05, r_max_diag=100.0,
)

fig, ax = plt.subplots(figsize=(8, 5))
plot_density_profiles_at_times(osc_inspect, ax=ax)
ax.set_title(rf"Energy density profiles — $\lambda_{{\rm GB}} = {lgb_inspect}$")
fig.tight_layout()
plt.show()

del sol_inspect

## Summary table

In [ ]:
print(f"{'lambda_GB':>12s}  {'C_max':>12s}  {'C/C_ref':>10s}")
print("-" * 38)
for lgb in sorted(results.keys()):
    Cmax = np.max(results[lgb]["osc"]["C"])
    ratio = Cmax / C_ref
    print(f"{lgb:12.4f}  {Cmax:12.6e}  {ratio:10.4f}")